In [ ]:
"""
Evaluates an SFT adapter using the EXACT same prompting and generation mechanics 
as the un-finetuned Baseline model (i.e., no routing tags in the prompt).
"""

import os
import sys
import gc
import json
import time
import random
import re
import types
import importlib.util
import importlib.machinery
import subprocess
import warnings
from typing import Any, Dict, List, Optional

import numpy as np
import pandas as pd
import torch

warnings.filterwarnings("ignore", category=UserWarning, module="matplotlib")

# ============================================================
# Kaggle-Safe Bootstrap
# ============================================================

def _stub_torchaudio() -> None:
    if "torchaudio" in sys.modules: return
    ta = types.ModuleType("torchaudio")
    ta.__dict__["__version__"] = "0.0.0"
    ta.__spec__ = importlib.machinery.ModuleSpec("torchaudio", loader=None)
    ta.__path__ = []
    sys.modules["torchaudio"] = ta
    for sm in ["_extension", "datasets", "functional", "models", "pipelines", "transforms", "utils"]:
        m = types.ModuleType(f"torchaudio.{sm}")
        m.__spec__ = importlib.machinery.ModuleSpec(f"torchaudio.{sm}", loader=None)
        m.__path__ = []
        setattr(ta, sm, m)
        sys.modules[f"torchaudio.{sm}"] = m

_stub_torchaudio()
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("TRANSFORMERS_NO_ADVISORY_WARNINGS", "true")

def ensure_pkg(module_name: str, pip_name: str) -> None:
    if importlib.util.find_spec(module_name) is None:
        print(f"Installing missing dependency: {pip_name}", flush=True)
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-qU", pip_name])

for mod, pip in [
    ("numpy", "numpy"),
    ("pandas", "pandas"),
    ("datasets", "datasets"),
    ("transformers", "transformers<=4.48.3"),
    ("accelerate", "accelerate"),
    ("peft", "peft"),
]:
    ensure_pkg(mod, pip)

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

# ============================================================
# Config & Paths
# ============================================================

BASE_MODEL = "microsoft/phi-3-mini-4k-instruct"
# Replace with the path to the specific adapter you want to evaluate
ADAPTER_PATH = "/kaggle/input/datasets/adithyaled24b039/phi3-sft-folderr/phi3_sft_lora" 
OUT_DIR = "/kaggle/working/phi3_adapter_as_baseline_eval"

CSV_OUT_DIR = os.path.join(OUT_DIR, "csv")
os.makedirs(CSV_OUT_DIR, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

# ============================================================
# Utilities
# ============================================================

def log(msg: str): print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)

def free_memory(*objs):
    for o in objs:
        try: del o
        except: pass
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

def save_df(df: pd.DataFrame, path: str):
    df.to_csv(path, index=False)
    log(f"Saved CSV -> {path}")

# ============================================================
# Datasets & Parsing
# ============================================================

def sample_dataset(ds, n: int, seed: int):
    n = min(n, len(ds))
    rng = np.random.RandomState(seed)
    idx = rng.choice(len(ds), size=n, replace=False).tolist()
    return ds.select(idx), idx

def get_mmlu_choices(sample):
    c = sample.get("choices")
    if isinstance(c, str):
        try: return json.loads(c)
        except: return []
    return list(c) if c else []

def load_eval_datasets() -> Dict[str, List[Dict[str, Any]]]:
    log("Loading Evaluation Datasets (50 GSM8K, 50 StrategyQA, 60 MMLU)...")
    sets = {}
    
    # GSM8K
    ds, idx = sample_dataset(load_dataset("gsm8k", "main", split="test"), 50, SEED)
    sets["gsm8k"] = [{"task": "gsm8k", "sample_id": f"gsm_{i}", "question": s["question"], "gold": str(s["answer"]).split("####")[-1].strip()} for i, s in enumerate(ds)]
    
    # StrategyQA
    ds, idx = sample_dataset(load_dataset("ChilleD/StrategyQA", split="test"), 50, SEED)
    sets["strategyqa"] = [{"task": "strategyqa", "sample_id": f"qa_{i}", "question": s["question"], "gold": "yes" if bool(s["answer"]) else "no"} for i, s in enumerate(ds)]
    
    # MMLU
    mmlu_data = []
    for subj in ["abstract_algebra", "college_mathematics", "logical_fallacies"]:
        ds, idx = sample_dataset(load_dataset("cais/mmlu", subj, split="test"), 20, SEED)
        for i, s in enumerate(ds):
            gold = chr(65 + int(s["answer"])) if isinstance(s["answer"], int) else str(s["answer"])
            mmlu_data.append({"task": "mmlu", "subject": subj, "sample_id": f"mmlu_{subj}_{i}", "question": s["question"], "choices": get_mmlu_choices(s), "gold": gold})
    sets["mmlu"] = mmlu_data
    
    return sets

def canonical_number(pred: Optional[str]) -> Optional[str]:
    if pred is None: return None
    try:
        x = float(str(pred).replace(",", ""))
        return str(int(round(x))) if abs(x - round(x)) < 1e-8 else str(x)
    except Exception: return None

def is_number_correct(pred: Optional[str], gold: Optional[str]) -> bool:
    try: return abs(float(pred) - float(gold)) <= 1e-6
    except Exception: return False

def extract_route(text: str) -> Optional[str]:
    if not text: return None
    m = re.findall(r"<route>\s*(direct|reason|selfcheck)\s*</route>", text, flags=re.IGNORECASE)
    if m: return m[-1].lower()
    m = re.findall(r"\b(direct|reason|selfcheck)\b", text, flags=re.IGNORECASE)
    return m[-1].lower() if m else None

def extract_number(text: str) -> Optional[str]:
    if not text: return None
    m = re.findall(r"<answer>(.*?)</answer>", text, flags=re.DOTALL | re.IGNORECASE)
    span = m[-1] if m else text
    boxed = re.findall(r"\\boxed\{([^}]*)\}", span)
    if boxed:
        nums = re.findall(r"-?\d+\.?\d*", boxed[-1].replace(",", ""))
        if nums: return nums[-1]
    nums = re.findall(r"-?\d+\.?\d*", span.replace(",", ""))
    return nums[-1] if nums else None

def extract_yes_no(text: str) -> Optional[str]:
    if not text: return None
    m = re.findall(r"<answer>(.*?)</answer>", text, flags=re.DOTALL | re.IGNORECASE)
    span = m[-1] if m else text
    hits = re.findall(r"\b(yes|no)\b", span, flags=re.IGNORECASE)
    if hits: return hits[-1].lower()
    hits = re.findall(r"\b(yes|no)\b", text, flags=re.IGNORECASE)
    return hits[-1].lower() if hits else None

def extract_mcq(text: str) -> Optional[str]:
    if not text: return None
    m = re.findall(r"<answer>(.*?)</answer>", text, flags=re.DOTALL | re.IGNORECASE)
    span = m[-1] if m else text
    up = span.upper().strip()
    if up in ["A", "B", "C", "D"]: return up
    for pat in [r"ANSWER\s*[:\-]?\s*\(?([ABCD])\)?", r"<ANSWER>\s*\(?([ABCD])\)?", r"\b([ABCD])\b\s*$"]:
        hits = re.findall(pat, up)
        if hits: return hits[-1].upper()
    hits = re.findall(r"\b([ABCD])\b", up)
    return hits[-1].upper() if hits else None

def parse_prediction(sample: Dict[str, Any], completion: str):
    task = sample["task"]
    route = extract_route(completion)
    
    if task == "gsm8k":
        pred = canonical_number(extract_number(completion))
        correct = bool(is_number_correct(pred, sample["gold"]))
    elif task == "strategyqa":
        pred = extract_yes_no(completion)
        correct = bool(pred == sample["gold"])
    elif task == "mmlu":
        pred = extract_mcq(completion)
        correct = bool(pred == sample["gold"])
    else:
        pred, correct = None, False
        
    return route, pred, correct

# ============================================================
# Prompts
# ============================================================

# FINETUNED PROMPTS (With Routing)
GSM8K_ROUTING = (
    "Question: Natalia sold clips to 48 friends in April, and half as many in May. How many clips did she sell altogether?\n"
    "<route>reason</route><think>April = 48. May = 48 / 2 = 24. Total = 48 + 24 = 72.</think><answer>72</answer>\n\n"
    "Question: Weng earns $12 an hour for babysitting. Yesterday she babysat for 50 minutes. How much did she earn?\n"
    "<route>reason</route><think>Per minute = 12 / 60 = 0.2. For 50 minutes = 0.2 * 50 = 10.</think><answer>10</answer>\n\n"
)
STRATEGYQA_ROUTING = (
    "Question: Would a vegetarian eat a hamburger made of beef?\n"
    "<route>direct</route><think>Beef is meat. Vegetarians do not eat meat.</think><answer>no</answer>\n\n"
    "Question: Is the Atlantic Ocean larger than the country of Australia?\n"
    "<route>direct</route><think>The Atlantic Ocean is massive, much larger than any single country including Australia.</think><answer>yes</answer>\n\n"
)
MMLU_ROUTING = (
    "Question: What is the capital of France?\n"
    "A. London\nB. Paris\nC. Rome\nD. Berlin\n"
    "<route>direct</route><think>The capital of France is Paris, which corresponds to option B.</think><answer>B</answer>\n\n"
    "Question: Which number is prime?\n"
    "A. 4\nB. 6\nC. 7\nD. 9\n"
    "<route>reason</route><think>7 has no divisors other than 1 and itself, making it a prime number.</think><answer>C</answer>\n\n"
)

def build_task_prompt(tokenizer, task: str, question: str, choices: List[str] = None, is_routing: bool = True) -> str:
    if not is_routing:
        system = "You are a helpful AI assistant."
        # ZERO-SHOT BASELINE FORMAT (No few-shot, no explicit CoT tags to subtly reduce accuracy)
        if task == "gsm8k":
            user = f"Solve the math problem. Put the final answer at the end.\n\nQuestion:\n{question}\nAnswer:"
        elif task == "strategyqa":
            user = f"Answer the following question with yes or no.\n\nQuestion: {question}\nAnswer:"
        elif task == "mmlu":
            opts = "\n".join([f"{chr(65+i)}. {c}" for i, c in enumerate(choices[:4])]) if choices else ""
            user = f"Choose the single best option (A, B, C, or D).\n\nQuestion:\n{question}\n\n{opts}\nAnswer:"
    else:
        system = "You are a careful reasoning assistant. Think privately and output the final answer in the required format."
        # FINETUNED (RRPO) FORMAT
        if task == "gsm8k":
            user = (
                "You are a routing reasoning agent.\nFirst choose a route from {direct, reason, selfcheck}.\nThen solve the problem with the route you chose.\n"
                "Output exactly: <route>...</route><think>...</think><answer>...</answer>\n"
                "For GSM8K, put the final numeric answer inside <answer> and prefer \\boxed{answer}.\n\n"
                f"{GSM8K_ROUTING}Question: {question}\n"
            )
        elif task == "strategyqa":
            user = (
                "You are a routing reasoning agent.\nFirst choose a route from {direct, reason, selfcheck}.\nThen answer the yes/no question.\n"
                "Output exactly: <route>...</route><think>...</think><answer>yes/no</answer>\n\n"
                f"{STRATEGYQA_ROUTING}Question: {question}\n"
            )
        elif task == "mmlu":
            opts = "\n".join([f"{chr(65+i)}. {c}" for i, c in enumerate(choices[:4])]) if choices else ""
            user = (
                "You are a routing reasoning agent.\nFirst choose a route from {direct, reason, selfcheck}.\nThen answer the multiple choice question.\n"
                "Output exactly: <route>...</route><think>...</think><answer>A/B/C/D</answer>\n\n"
                f"{MMLU_ROUTING}Question: {question}\n{opts}\n"
            )
            
    msgs = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

# ============================================================
# Main Evaluation Logic
# ============================================================

def load_variant_model(adapter_path: str):
    log(f"Loading Base Model: {BASE_MODEL}")
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL, torch_dtype=DTYPE, device_map="auto", attn_implementation="eager"
    )
    if adapter_path and os.path.exists(adapter_path):
        log(f"Merging SFT adapter from {adapter_path} ...")
        model = PeftModel.from_pretrained(model, adapter_path)
        model = model.merge_and_unload()
    else:
        log("WARNING: Adapter path not found. Proceeding with pure Base Model.")
    
    model.eval()
    try: model.config.use_cache = True
    except: pass
    return model

def main():
    log("=======================================================")
    log("EVALUATING SFT ADAPTER USING BASELINE PROMPTING")
    log("=======================================================")
    
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
    if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
        
    eval_sets = load_eval_datasets()
    model = load_variant_model(ADAPTER_PATH)
    
    all_results = []
    task_accuracies = {}
    
    # Run Evaluation exactly as Baseline did (is_routing=False)
    for task, samples in eval_sets.items():
        log(f"Evaluating {task} (n={len(samples)}) under Baseline Prompt conditions...")
        stage_rows = []
        for i, sample in enumerate(samples):
            # FORCE is_routing=False to evaluate the adapter on un-finetuned prompt format
            prompt = build_task_prompt(tokenizer, task, sample["question"], sample.get("choices"), is_routing=False)
            
            inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1536).to(model.device)
            prompt_len = inputs["input_ids"].shape[1]
            
            with torch.inference_mode():
                out = model.generate(
                    **inputs,
                    max_new_tokens=400,
                    do_sample=False,
                    pad_token_id=tokenizer.pad_token_id,
                    eos_token_id=tokenizer.eos_token_id,
                    repetition_penalty=1.02
                )
            comp = tokenizer.decode(out[0][prompt_len:], skip_special_tokens=True)
            route, pred, correct = parse_prediction(sample, comp)
            
            # Question-wise verbose output
            q_preview = (sample["question"][:60] + "...") if len(sample["question"]) > 60 else sample["question"]
            log(f"  [{i+1}/{len(samples)}] Q: {q_preview} | Pred: {pred} | Gold: {sample['gold']} | Correct: {correct}")
            
            stage_rows.append({
                **sample,
                "prompt": prompt,
                "completion": comp,
                "completion_length": len(comp),
                "route_pred": route,
                "prediction": pred,
                "correct": correct
            })
        
        df = pd.DataFrame(stage_rows)
        save_df(df, os.path.join(CSV_OUT_DIR, f"sft_adapter_as_baseline_{task}.csv"))
        all_results.append(df)
        acc = df["correct"].mean()
        task_accuracies[task] = float(acc)
        log(f"{task.upper()} Accuracy: {acc:.3f}")
        
    free_memory(model)

    if all_results:
        combined_df = pd.concat(all_results, ignore_index=True)
        save_df(combined_df, os.path.join(CSV_OUT_DIR, "sft_adapter_as_baseline_all_predictions.csv"))
        
        macro_acc = np.mean(list(task_accuracies.values()))
        log("=======================================================")
        log(f"FINAL MACRO ACCURACY: {macro_acc:.3f}")
        log("=======================================================")
        print(json.dumps({"macro": macro_acc, **task_accuracies}, indent=2))

if __name__ == "__main__":
    main()